# 01 – Data Audit and Cleaning

**Project:** Developer Salary Prediction  
**Dataset:** Stack Overflow Developer Survey 2025 ([survey.stackoverflow.co](https://survey.stackoverflow.co/), license: ODbL 1.0)  
**Status:** In progress. This notebook currently covers loading the data, a first look, the data dictionary, the target variable, the scope decision, missing values per column, the scope filters, the data quality audit and the first cleaning step. Further cleaning steps will be added in the next sessions.

### Purpose of this notebook
Before any cleaning or modelling, we answer four basic questions about the data:

1. What does one row represent, and how much data do we have?
2. What is the state of the target variable?
3. Which rows belong to our business problem?
4. Which columns can be used as features, and which ones are risky (leakage)?

The order matters: each answer affects the next one. For example, if most target values are missing, the scope decision must take that into account.

### Business problem (from the project definition)
A software developer who receives a job offer wants to know whether the salary is fair compared with peers who have a similar profile. The model will predict a typical annual salary for a given developer profile.

## 0. Setup

In [1]:
import pandas as pd

# Show long text (e.g. survey questions) in full instead of cutting it off
pd.set_option("display.max_colwidth", None)

## 1. Load the data

Three files from Stack Overflow are stored in `data/raw/` (not uploaded to GitHub because the results file is 135 MB):

| File | Content |
|---|---|
| `survey_results_2025.csv` | The survey answers: one row per respondent |
| `survey_schema_2025.csv` | Data dictionary: which column belongs to which survey question |
| `survey_questionnaire_2025.pdf` | The full questionnaire, including all answer options |

**Why `low_memory=False`?** pandas reads large files in chunks and guesses each column's data type per chunk. When a column contains numbers in one chunk and text in another, pandas cannot decide and shows a `DtypeWarning`. With `low_memory=False`, pandas reads the whole column before deciding the type.

In [2]:
# Paths are relative to the notebooks/ folder: "../" goes up to the repository root
df = pd.read_csv("../data/raw/survey_results_2025.csv", low_memory=False)
schema = pd.read_csv("../data/raw/survey_schema_2025.csv")

### Finding 1: mixed-type columns
When the file was first loaded without `low_memory=False`, pandas raised a `DtypeWarning` for 11 columns (positions 56, 74, 92, 97, 98, 105, 109, 110, 132, 162, 165). The code below shows their names.

This is our first data quality finding: these columns contain inconsistent data types. They turn out to be free-text "Other (please specify)" type answers, so they are unlikely to be used as model features.

In [3]:
# Column positions reported in the DtypeWarning
mixed_type_positions = [56, 74, 92, 97, 98, 105, 109, 110, 132, 162, 165]

df.columns[mixed_type_positions].tolist()

['JobSatPoints_15_TEXT',
 'DatabaseHaveEntry',
 'DevEnvHaveEntry',
 'SOTagsHaveEntry',
 'SOTagsWant Entry',
 'OfficeStackWantEntry',
 'CommPlatformHaveEntr',
 'CommPlatformWantEntr',
 'SO_Actions_15_TEXT',
 'AIAgentOrchestration',
 'AIAgentObsWrite']

## 2. First look at the data

**Question 1: What does one row represent?** One row is one survey respondent.

- `df.shape` tells us how many respondents (rows) and how many survey fields (columns) we have.
- `df.dtypes.value_counts()` summarises how many columns are numeric and how many are text. This matters because text columns must be encoded (e.g. with `OneHotEncoder`) before they can be used in a model.

*Why not `df.info()`?* It gives the same information, but prints one line per column. With 172 columns the output becomes too long to read.

In [4]:
df.head()

,ResponseId,MainBranch,Age,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,LearnCodeAI,...,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,"Caring for dependents (children, elderly, etc.)",8.0,"Yes, I am not new to coding but am learning new coding techniques or programming language","Online Courses or Certification (includes all media types);Other online resources (e.g. standard search, forum, online community)","Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",...,Vertex AI,NaN,NaN,NaN,ChatGPT,NaN,When I don’t trust AI’s answers,"Troubleshooting, profiling, debugging",61256.0,10.0
1,2,I am a developer by profession,25-34 years old,"Associate degree (A.A., A.S., etc.)",Employed,NaN,2.0,"Yes, I am not new to coding but am learning new coding techniques or programming language","Online Courses or Certification (includes all media types);Other online resources (e.g. standard search, forum, online community);Books / Physical media;Videos (not associated with specific online course or certification);Stack Overflow or Stack Exchange","Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers;When I want to fully understand something;When I want to learn best practices;When I want to compare different solutions;When I have ethical or security concerns about code,All skills. AI is a flop.,104413.0,9.0
2,3,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Independent contractor, freelancer, or self-employed",None of the above,10.0,"Yes, I am not new to coding but am learning new coding techniques or programming language",Online Courses or Certification (includes all media types);Videos (not associated with specific online course or certification);Technical documentation (is generated for/by the tool or system),"Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",...,NaN,NaN,NaN,NaN,ChatGPT;Claude Code;GitHub Copilot;Google Gemini,NaN,When I don’t trust AI’s answers;When I want to fully understand something;When I want to learn best practices;When I have ethical or security concerns about code;When I’m stuck and can’t explain the problem,"Understand how things actually work, problem solving, and algorithms",53061.0,8.0
3,4,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,None of the above,4.0,"Yes, I am not new to coding but am learning new coding techniques or programming language","Other online resources (e.g. standard search, forum, online community);Videos (not associated with specific online course or certification);Stack Overflow or Stack Exchange;Technical documentation (is generated for/by the tool or system);AI CodeGen tools or AI-enabled apps","Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",...,NaN,NaN,NaN,NaN,ChatGPT;Claude Code,NaN,When I don’t trust AI’s answers;When I want to fully understand something;When I have ethical or security concerns about code;When I’m stuck and can’t explain the problem,NaN,36197.0,6.0
4,5,I am a developer by profession,35-44 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Independent contractor, freelancer, or self-employed","Caring for dependents (children, elderly, etc.)",21.0,"No, I am not new to coding and did not learn new coding techniques or programming languages",NaN,"Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers,"critical thinking, the skill to define the task correctly",60000.0,7.0


In [5]:
# Number of rows (respondents) and columns (survey fields)
print(df.shape)

# How many columns of each data type
df.dtypes.value_counts()

(49191, 172)


object     119
float64     52
int64        1
Name: count, dtype: int64

In [6]:
# ResponseId should be unique: one ID per respondent
df["ResponseId"].duplicated().sum()

np.int64(0)

## 3. How to look up a column

Three sources explain the columns:

1. **The schema file** gives the question text for each column. Important fields: `qname` (column name), `question` (question text) and `type` (`MC` = multiple choice, `TE` = text entry, `RO` = rank order, `Matrix` = a table of sub-questions).
2. **Matrix questions are split into several columns.** For example, the schema has one question called `Language`, but the data has `LanguageHaveWorkedWith`, `LanguageWantToWorkWith` and `LanguageAdmired`. To find these, search the schema for the beginning of the column name.
3. **Two columns were not asked in the survey.** `ResponseId` is a respondent ID, and `ConvertedCompYearly` was calculated by Stack Overflow from `CompTotal` and `Currency` (salary converted to USD).

For answer options (e.g. the categories of `OrgSize`), use the questionnaire PDF.

In [7]:
# Look up one column by its exact name
schema[schema["qname"] == "WorkExp"][["qname", "question", "type"]]

,qname,question,type
54,WorkExp,"How many years of professional work experience do you have? Please round to the nearest whole number, excluding any decimal points. If your answer is '0', please leave blank.",TE


In [8]:
# Search for all columns whose name contains a keyword
schema[schema["qname"].str.contains("Comp")][["qname", "question", "type"]]

,qname,question,type
74,CompTotal,"What is your current total <b>annual</b> compensation (salary, bonuses, and perks, before taxes and deductions) in terms of your day-to-day currency? Please enter a whole number in the box below, without any punctuation. If you are paid hourly, please estimate an equivalent yearly salary. If you prefer not to answer, please leave the box empty.",TE
119,AIComplex,How well do the AI tools you use in your development workflow handle complex tasks?,MC


## 4. Data dictionary

The columns we currently plan to use, based on the schema file and the questionnaire PDF. **Role** shows how each column is used: target, scope filter, candidate feature or excluded because of leakage.

| Column | Meaning | Type | Role | Notes |
|---|---|---|---|---|
| `ConvertedCompYearly` | Total annual compensation (salary, bonuses, perks, before tax), converted to USD | Numeric | **Target** | Not asked directly: calculated by Stack Overflow from `CompTotal` and `Currency`. Answering was optional |
| `CompTotal` | Total annual compensation in the respondent's own currency | Numeric | **Excluded (leakage)** | Contains the answer the model must predict |
| `Currency` | Currency used day-to-day | Categorical (single choice) | **Excluded (leakage)** | Used to calculate the target |
| `MainBranch` | Whether the respondent is a professional developer, learner, hobbyist, etc. | Categorical (single choice) | Scope filter | Kept: "I am a developer by profession" (our target user is a professional developer) |
| `Employment` | Current employment status | Categorical (single choice) | Scope filter | Kept: "Employed". Options: Employed, Independent contractor/freelancer, Student, Not employed, Retired, Prefer not to say. There is no full-time/part-time split in 2025 |
| `Country` | Country of residence | Categorical (single choice) | Scope filter / candidate feature | 177 countries in the raw data. Kept: EU-27 plus Iceland, Norway, Switzerland and the United Kingdom (31 countries). Salary levels differ strongly by country, so `Country` is also a feature |
| `WorkExp` | Years of professional work experience | Numeric | Candidate feature | The question says: *"If your answer is '0', please leave blank"*, so some missing values mean 0 years |
| `YearsCode` | Total years of coding, **including education** | Numeric | Candidate feature | Usually higher than `WorkExp` because it includes study years |
| `DevType` | Current job role (e.g. back-end, data scientist, engineering manager) | Categorical (single choice) | Candidate feature | Many categories: rare ones may need grouping |
| `EdLevel` | Highest completed level of formal education | Categorical (single choice) | Candidate feature | Includes an "Other" option |
| `OrgSize` | Number of employees at the employer | Categorical (single choice) | Candidate feature | Ordered categories; includes "I don't know" |
| `RemoteWork` | Current work situation | Categorical (single choice) | Candidate feature | 5 options, including two different "Hybrid" options and "Your choice" |
| `Industry` | Industry of the employer | Categorical (single choice) | Candidate feature | Includes an "Other" option |
| `LanguageHaveWorkedWith` | Programming languages used in the past year | Multi-select text | Candidate feature | Several answers in one cell, separated by `;` (e.g. `Python;SQL`). Must be split before modelling |
| `ICorPM` | Individual contributor or people manager | Categorical (two values) | Candidate feature | Manager status is a strong salary driver and known by the user. Suitable for a two-group test (`ttest_ind`) |
| `Age` | Age group (e.g. "25-34 years old") | Ordered categorical | Candidate feature (to verify) | Asked as age ranges, not exact age. May overlap strongly with `WorkExp`: check in EDA before keeping it |

This table will be updated when the scope and the final feature list are decided.

## 5. Target variable: `ConvertedCompYearly`

**Question 2: What is the state of the target?**

- **Missing values:** The salary question was optional. Rows without a target value cannot be used to train the model, so the number of missing values tells us how many rows we can really work with. The percentage makes the number easier to judge.
- **Distribution (`describe()`):** What to look for:
  - *Minimum and maximum:* unrealistic values (e.g. 1 USD or tens of millions) are probably input errors and will be handled during cleaning.
  - *Mean vs. median (50%):* if the mean is much higher than the median, a few very high salaries pull the average up. This means the distribution is right-skewed, which matters for EDA and model choice.

In [11]:
target = "ConvertedCompYearly"

# Count and share of missing target values
print("Missing:", df[target].isnull().sum())
print("Missing %:", round(df[target].isnull().mean() * 100, 1))  # mean() of True/False = share of missing

# Summary statistics of the target
df[target].describe().round(0)

Missing: 25244
Missing %: 51.3


count       23947.0
mean       101762.0
std        461757.0
min             1.0
25%         38171.0
50%         75320.0
75%        120596.0
max      50000000.0
Name: ConvertedCompYearly, dtype: float64

**Observations:**
- **Missing target:** 25,244 respondents (51.3%) did not report a salary. 23,947 rows remain usable, which is enough for a regression model. Rows without a target will be removed: imputing the target would teach the model invented answers.
- **Typical range:** The middle 50% earn between about 38,000 and 121,000 USD per year (median 75,320 USD), which is realistic for developer salaries.
- **Unrealistic extremes:** The minimum (1 USD) and maximum (50,000,000 USD) are not credible. The very large standard deviation (about 4.5 times the mean) confirms that a few extreme values distort the statistics. An outlier rule will be defined in the cleaning step.
- **Right-skewed distribution:** The mean (101,762 USD) is clearly higher than the median, so a few very high salaries pull the average up. The median is a better summary, and a log-transformation of the target will be tested during modelling.

## 6. Columns that define the scope

**Question 3: Which rows belong to our business problem?**

The answer comes from the business problem, not from the code. Our target user is a developer evaluating a job offer, so the model must learn from people who are comparable to that user:

- **`MainBranch`:** The survey includes students, hobbyists and former developers. Their answers are not a fair comparison for someone receiving a job offer.
- **`Employment`:** The annual income of an employee and a freelancer are not comparable in the same way.
- **`Country`:** Salary levels differ a lot between countries. Before choosing world, Europe or the Netherlands, we need to see whether each option has enough respondents.

We look at these distributions **only for rows with a salary**, because rows without a target cannot be used anyway. "How many respondents from the Netherlands?" is less useful than "How many respondents from the Netherlands shared their salary?"

In [10]:
# Keep only respondents who reported a salary
has_salary = df[df[target].notnull()]

print(has_salary["MainBranch"].value_counts(), "\n")
print(has_salary["Employment"].value_counts(), "\n")

# Top 20 countries by number of respondents with a salary
has_salary["Country"].value_counts().head(20)

MainBranch
I am a developer by profession                                                                20195
I am not primarily a developer, but I write code sometimes as part of my work/studies          2187
I used to be a developer by profession, but no longer am                                        498
I work with developers or my work supports developers but am not a developer by profession      390
I am learning to code                                                                           363
I code primarily as a hobby                                                                     314
Name: count, dtype: int64 

Employment
Employed                                                19500
Independent contractor, freelancer, or self-employed     3050
Student                                                   703
Not employed                                              504
Retired                                                   133
I prefer not to say                     

Country
United States of America                                5243
Germany                                                 2141
United Kingdom of Great Britain and Northern Ireland    1486
India                                                   1096
France                                                  1027
Canada                                                   915
Ukraine                                                  709
Brazil                                                   639
Netherlands                                              619
Poland                                                   589
Italy                                                    581
Australia                                                569
Spain                                                    560
Sweden                                                   471
Switzerland                                              418
Czech Republic                                           361
Austria         

**Observations:**
- **MainBranch:** 84% of respondents with a salary (20,195) are professional developers. Other groups (occasional coders, former developers, learners, hobbyists) do not represent our target user, so only "I am a developer by profession" is kept.
- **Employment:** 19,500 are employed and 3,050 are freelancers. Freelance income is calculated differently and is not comparable with a salary offer, so only "Employed" is kept.
- **Country:** The United States dominates (5,243, more than twice Germany). The Netherlands has 619 respondents with a salary.

**Data quality notes (for the audit table):**
- Some respondents who are "Not employed" (504) or "Student" (703) still reported a salary, probably from a previous or part-time job. These rows are removed by the Employment filter.
- Some countries appear under two names, e.g. "Moldova" and "Republic of Moldova", or "South Korea" and "Republic of Korea" (inconsistent string values). These countries fall outside the chosen scope, so they do not affect the model.

**Scope decision:** Professional developers (MainBranch) who are employed (Employment) and live in Europe, defined as the **EU-27 plus Iceland, Norway, Switzerland and the United Kingdom** (31 countries). These countries largely form one labour market in which developers can move for work.
- *World* was rejected: the US dominates the data and most of the ~150 countries have very few rows.
- *Netherlands only* was rejected: about 450 rows, so the test set would be too small for reliable results.
- *Europe* keeps enough data (about 7,700 rows) and matches a realistic labour market for a Netherlands-based user. Salary differences between countries are handled by using `Country` as a feature.

## 7. Missing values per column

**Question 4 (first part): Which columns are usable?** Not all 172 columns matter. A column can be dropped for two different reasons:

1. **It is not relevant to the business problem** (e.g. opinions about AI tools or how often someone visits Stack Overflow). This is decided with business logic, as in the data dictionary.
2. **It is too empty to be useful.** This is checked with the data, below.

The two reasons can give different answers: a relevant column with some missing values can still be valuable, and a complete but irrelevant column is still dropped. So there is no single rule like "drop every column with missing values".

How the calculation works: `isnull()` marks empty cells as True, `.mean()` gives the share of True values per column (True = 1, False = 0), `.mul(100).round(1)` turns it into a percentage, and `sort_values(ascending=False)` puts the emptiest columns first.

In [ ]:
# Share of missing values per column, in percent, highest first
missing_pct = df.isnull().mean().mul(100).round(1).sort_values(ascending=False)

missing_pct.head(30)

In [ ]:
# How many columns pass a given emptiness threshold
print("Columns more than 50% empty:", (missing_pct > 50).sum())
print("Columns more than 90% empty:", (missing_pct > 90).sum())

What matters most is how complete our **candidate features** are among the rows we can actually use for training, i.e. respondents **with a salary**.

In [ ]:
candidate_features = ["WorkExp", "YearsCode", "Country", "DevType", "EdLevel",
                      "OrgSize", "RemoteWork", "Industry", "LanguageHaveWorkedWith",
                      "ICorPM", "Age"]

# Missing % of candidate features, only among respondents with a salary
has_salary[candidate_features].isnull().mean().mul(100).round(1).sort_values(ascending=False)

**Observations:**
- **71 columns are more than 50% empty and 30 columns are more than 90% empty.** Most of them end in `Entry`, `Write` or `_TEXT`.
- **This is mainly caused by the structure of the survey, not by bad data.** There are three sources of missing values:
  1. *Free-text boxes* ("Other, please specify"): most respondents choose from the list, so these stay empty.
  2. *Conditional questions*: some questions were only shown to certain respondents. For example, the questions on AI agent tools start with "You indicated you use or develop AI agents...", so people who do not use agents never saw them.
  3. *Optional questions*: such as the salary question.
- **The candidate features are fairly complete** among respondents with a salary: the emptiest ones (`ICorPM`, `OrgSize`, `RemoteWork`) are only around 12–13% empty. The very empty columns are columns we do not plan to use.
- Missing rates can change after filtering, because some questions mainly apply to employed people. We check the candidate features again on the final scope in Section 8.
- **`WorkExp` is only about 2% empty** among respondents with a salary, so the rule "leave blank if the answer is 0" affects very few rows.

## 8. Apply the scope filters

The filters follow the scope decision in Section 6. We apply them one step at a time and record the number of rows after each step. This makes the impact of every decision visible, which the rubric asks for ("explain how cleaning decisions affect the analysis").

Before filtering by country, we check that every country name in our Europe list exactly matches the spelling in the data. Country names in this survey are long and sometimes unusual (e.g. "United Kingdom of Great Britain and Northern Ireland"). A spelling mismatch would not cause an error: the country would silently disappear from the data.

In [ ]:
# Europe = EU-27 plus Iceland, Norway, Switzerland and the United Kingdom (31 countries)
eu_27 = ["Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", "Czech Republic", "Denmark",
         "Estonia", "Finland", "France", "Germany", "Greece", "Hungary", "Ireland", "Italy",
         "Latvia", "Lithuania", "Luxembourg", "Malta", "Netherlands", "Poland", "Portugal",
         "Romania", "Slovakia", "Slovenia", "Spain", "Sweden"]
other_europe = ["Iceland", "Norway", "Switzerland",
                "United Kingdom of Great Britain and Northern Ireland"]
europe = eu_27 + other_europe

print("Countries in the list:", len(europe))

# Safety check: names in our list that do not appear in the data (should be an empty list)
[country for country in europe if country not in df["Country"].unique()]

In [ ]:
# Step 1: keep rows with a salary (the target cannot be imputed)
step1 = df[df[target].notnull()]

# Step 2: keep professional developers only
step2 = step1[step1["MainBranch"] == "I am a developer by profession"]

# Step 3: keep employed respondents only (no freelancers, students, unemployed, retired)
step3 = step2[step2["Employment"] == "Employed"]

# Step 4: keep respondents living in Europe (31 countries)
df_scope = step3[step3["Country"].isin(europe)].copy()

# Funnel table: rows remaining after each filter
funnel = pd.DataFrame({
    "step": ["Raw data", "Has salary", "Professional developer", "Employed", "Lives in Europe"],
    "rows": [len(df), len(step1), len(step2), len(step3), len(df_scope)]
})
funnel["share_of_raw_%"] = (funnel["rows"] / len(df) * 100).round(1)

# Rows removed by each step, and the loss compared with the previous step
# shift(1) moves the column one row down, so each row can see the previous step's value
funnel["removed"] = funnel["rows"].shift(1) - funnel["rows"]
funnel["loss_vs_previous_%"] = (funnel["removed"] / funnel["rows"].shift(1) * 100).round(1)
funnel

`.copy()` creates an independent copy of the filtered data. Without it, later changes to `df_scope` could trigger pandas' `SettingWithCopyWarning`, because pandas cannot tell whether we want to change the filtered rows only or the original `df` as well.

In [ ]:
# Respondents per country in the final scope
# to_string() prints the full list instead of cutting off the middle
print("Countries:", df_scope["Country"].nunique())
print(df_scope["Country"].value_counts().to_string())

In [ ]:
# Missing % of candidate features within the final scope
df_scope[candidate_features].isnull().mean().mul(100).round(1).sort_values(ascending=False)

**Observations:**
- **Validation check:** All 31 country names in the Europe list match the spelling in the data (empty list). Without this check, a misspelled name would have silently removed that country.
- **Funnel:** 7,709 rows remain (15.7% of the raw data). Two kinds of loss should be distinguished:
  - *Quality-driven:* 25,244 rows (51.3%) have no salary and cannot be used.
  - *Scope-driven:* the next three filters remove healthy data on purpose because these respondents do not match the target user. The largest scope step is the Europe filter (54.4% of the remaining rows).
- **Model validity:** Because of these filters, the model is only valid for employed professional developers living in Europe. It must not be used for freelancers or developers outside Europe. This limitation will be stated in the README and slides.
- **Country concentration:** Germany and the United Kingdom form 34.5% of the data; together with France and the Netherlands, four countries form almost half (49.4%). The Netherlands, our user's country, is well represented (446 rows, 4th place).
- **Small countries:** 8 countries have fewer than 50 respondents (208 rows, 2.7% of the data), Iceland only 6. Their statistics are unreliable and they may be missing from the train or test set. A grouping decision will be made during the cleaning of categorical columns.
- **Missing values in the final scope:** All candidate features are less than 6% empty. Missing rates of `ICorPM`, `RemoteWork` and `OrgSize` dropped from about 12–13% to under 1.5% after filtering, probably because people outside the scope (e.g. freelancers, students) skipped these work-related questions. This confirms that missing-value decisions should be made on the final scope.
- **Remaining gaps:** `LanguageHaveWorkedWith` (5.9%) may be empty because some respondents were not shown the language list after a screening question. `ICorPM` (1.4%) and `WorkExp` (0.6%) will be handled in the next cleaning steps.

## 9. Data quality audit

From here on, all checks are done on the filtered data (`df_scope`), because missing-value and cleaning decisions should be made on the final scope (see Section 8).

The rubric asks for a **systematic** check of data quality. It lists the issues to look for: missing values, duplicate records, unnecessary columns, inconsistent string values, incorrect formatting and incorrect data types. We first **detect** all issues in one place, and only then **fix** them one by one in the next sections. Detecting before fixing prevents a hidden problem from distorting the analysis later.

This section has four steps:
1. Remove unnecessary columns
2. Build an audit table with one row per column
3. Check for duplicate records
4. Check the values of the categorical columns for inconsistencies

### 9.1 Remove unnecessary columns

We only keep the columns we need:
- `ResponseId`: not a feature, but useful to trace a row back to the original data.
- The target `ConvertedCompYearly`.
- The candidate features from the data dictionary.

Everything else is dropped. Note two groups of dropped columns:
- **Leakage columns** (`CompTotal`, `Currency`): they contain the answer the model must predict.
- **Scope filter columns** (`MainBranch`, `Employment`): after filtering they have only one value for every row. A column with the same value everywhere carries no information and cannot help the model.

All cleaning steps from now on are applied to a new DataFrame, `df_clean`, so `df_scope` stays unchanged as a reference.

In [19]:
# Columns needed for the analysis: ID, target and candidate features
columns_to_keep = ["ResponseId", target] + candidate_features

# New DataFrame for all cleaning steps; .copy() keeps df_scope unchanged
df_clean = df_scope[columns_to_keep].copy()

print(df_clean.shape)

(7709, 13)


### 9.2 Audit table

The audit table summarises every column in one row, so all issues can be seen at a glance:

| Column of the table | What it shows | What to look for |
|---|---|---|
| `dtype` | The data type pandas assigned | Numbers stored as text, or text that should be a number |
| `missing_%` | Share of empty cells | Columns that need a missing-value decision |
| `unique_values` | Number of different values | Very many categories (hard to encode) or only one value (useless) |
| `example_values` | The first three different values | Formatting problems, unexpected values, multi-select answers |

How `example_values` is built: for each column, `dropna()` removes empty cells, `unique()` keeps the different values, and `[:3]` takes the first three. The list comprehension (`[... for col in df_clean.columns]`) repeats this for every column.

In [20]:
audit = pd.DataFrame({
    "dtype": df_clean.dtypes.astype(str),
    "missing_%": df_clean.isnull().mean().mul(100).round(1),
    "unique_values": df_clean.nunique(),
    "example_values": [df_clean[col].dropna().unique()[:3].tolist() for col in df_clean.columns]
})

audit

,dtype,missing_%,unique_values,example_values
ResponseId,int64,0.0,7709,"[2, 28, 36]"
ConvertedCompYearly,float64,0.0,1843,"[104413.0, 209643.0, 95132.0]"
WorkExp,float64,0.6,49,"[2.0, 24.0, 10.0]"
YearsCode,float64,0.2,54,"[10.0, 36.0, 25.0]"
Country,object,0.0,31,"[Netherlands, Sweden, Germany]"
DevType,object,0.0,31,"[Developer, back-end, Developer, full-stack, Developer, game or graphics]"
EdLevel,object,0.1,8,"[Associate degree (A.A., A.S., etc.), Bachelor’s degree (B.A., B.S., B.Eng., etc.), Master’s degree (M.A., M.S., M.Eng., MBA, etc.)]"
OrgSize,object,0.1,9,"[500 to 999 employees, 1,000 to 4,999 employees, Less than 20 employees]"
RemoteWork,object,0.3,5,"[Hybrid (some in-person, leans heavy to flexibility), Hybrid (some remote, leans heavy to in-person), Your choice (very flexible, you can come in when you want or just as needed)]"
Industry,object,0.1,15,"[Retail and Consumer Services, Banking/Financial Services, Higher Education]"


### 9.3 Duplicate records

Two different checks:
1. **Duplicate `ResponseId`:** the same ID appearing twice would be a technical error. We checked this on the raw data in Section 2; we confirm it again here.
2. **Duplicate answers:** two rows with different IDs but identical answers in every other column. This could be the same person submitting the survey twice. To check this, we temporarily drop `ResponseId` with `drop(columns=...)`, because different IDs would otherwise always make rows look different.

Identical answers can also happen by coincidence. Whether duplicates should be removed depends on how likely a coincidence is, so we look at the rows before deciding.

In [21]:
# 1. The same ID twice (technical error)
print("Duplicate ResponseId:", df_clean["ResponseId"].duplicated().sum())

# 2. Different IDs, but identical answers in all other columns
print("Duplicate answers (excluding ResponseId):",
      df_clean.drop(columns="ResponseId").duplicated().sum())

Duplicate ResponseId: 0
Duplicate answers (excluding ResponseId): 3


In [22]:
# Show all rows that take part in a duplicate pair (keep=False marks every copy, not only the second one)
answer_cols = df_clean.columns.drop("ResponseId")
df_clean[df_clean.duplicated(subset=answer_cols, keep=False)].sort_values(target)

,ResponseId,ConvertedCompYearly,WorkExp,YearsCode,Country,DevType,EdLevel,OrgSize,RemoteWork,Industry,LanguageHaveWorkedWith,ICorPM,Age
33769,33770,28083.0,9.0,8.0,Czech Republic,"Developer, back-end","Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","1,000 to 4,999 employees","Hybrid (some remote, leans heavy to in-person)",Higher Education,Java;SQL,Individual contributor,25-34 years old
37400,37401,28083.0,9.0,8.0,Czech Republic,"Developer, back-end","Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","1,000 to 4,999 employees","Hybrid (some remote, leans heavy to in-person)",Higher Education,Java;SQL,Individual contributor,25-34 years old
45690,45691,38285.0,27.0,30.0,Portugal,"Developer, embedded applications or devices","Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","10,000 or more employees","Hybrid (some remote, leans heavy to in-person)",Other:,Bash/Shell (all shells);C;C++;MicroPython;Python,Individual contributor,45-54 years old
45712,45713,38285.0,27.0,30.0,Portugal,"Developer, embedded applications or devices","Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","10,000 or more employees","Hybrid (some remote, leans heavy to in-person)",Other:,Bash/Shell (all shells);C;C++;MicroPython;Python,Individual contributor,45-54 years old
13288,13289,93972.0,16.0,33.0,Germany,"Developer, full-stack","Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","1,000 to 4,999 employees","Hybrid (some in-person, leans heavy to flexibility)",Insurance,C#;HTML/CSS;JavaScript;PowerShell;SQL;TypeScript,Individual contributor,35-44 years old
43909,43910,93972.0,16.0,33.0,Germany,"Developer, full-stack","Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","1,000 to 4,999 employees","Hybrid (some in-person, leans heavy to flexibility)",Insurance,C#;HTML/CSS;JavaScript;PowerShell;SQL;TypeScript,Individual contributor,35-44 years old


#### Are the duplicates the same person?

The three pairs are identical in all 12 analysis columns, including very specific details (e.g. the same combination of five programming languages, when there are 4,397 different combinations in the data). This suggests the same person submitted the survey twice. On the other hand, a coincidence is possible: the Czech pair works in higher education, where salaries often follow fixed pay scales, and uses a very common language combination.

To collect more evidence, we compare the pairs on **all 172 original columns**. If the same person submitted twice, most other answers should match as well.

In [25]:
# IDs of all rows that are part of a duplicate pair
dup_ids = df_clean.loc[df_clean.duplicated(subset=answer_cols, keep=False), "ResponseId"]

# The same rows with all original columns
dup_full = df_scope[df_scope["ResponseId"].isin(dup_ids)]
full_cols = df_scope.columns.drop("ResponseId")

# How many of these rows are identical on ALL columns?
print("Rows identical on all columns:", dup_full.duplicated(subset=full_cols, keep=False).sum())

Rows identical on all columns: 0


In [26]:
# Compare one pair in detail: in how many columns do the two answers differ?
row_a = df_scope[df_scope["ResponseId"] == 33770].iloc[0].drop("ResponseId")
row_b = df_scope[df_scope["ResponseId"] == 37401].iloc[0].drop("ResponseId")

# Two empty cells (NaN) are not equal in pandas, so we treat "both empty" as the same answer
differs = ~((row_a == row_b) | (row_a.isna() & row_b.isna()))

print("Columns with different answers:", differs.sum())
differs[differs].index.tolist()[:10]

Columns with different answers: 48


['TechEndorse_2',
 'TechEndorse_3',
 'TechEndorse_8',
 'JobSatPoints_3',
 'JobSatPoints_13',
 'JobSatPoints_14',
 'PlatformChoice',
 'PlatformHaveWorkedWith',
 'PlatformWantToWorkWith',
 'PlatformAdmired']

**Decision:** None of the pairs are identical on all columns: each pair differs in about 50 columns. Most of these are opinion and ranking questions (e.g. why a technology is endorsed), which the same person could answer differently on a second attempt. So the evidence is not conclusive.

When the evidence is not conclusive, we compare the cost of each option:
- **Removing the second copy** loses 3 rows (0.04% of the data). Even if they are different people, this cannot noticeably change the results.
- **Keeping both copies** creates a small leakage risk: if one copy ends up in the training set and the other in the test set, the model is tested on an answer it has already seen.

Removing has almost no cost, keeping has a small risk, so the second copy of each pair is removed (`keep="first"`).

In [27]:
rows_before = len(df_clean)

# Keep the first copy of each duplicate pair, remove the second
df_clean = df_clean.drop_duplicates(subset=answer_cols, keep="first")

print("Rows before:", rows_before)
print("Rows after: ", len(df_clean))
print("Removed:    ", rows_before - len(df_clean))

Rows before: 7709
Rows after:  7706
Removed:     3


### 9.4 Values of the categorical columns

The audit table only shows three example values per column. To find **inconsistent string values**, we need to see every category of each single-choice column. `value_counts(dropna=False)` also counts empty cells (`NaN`), so missing values are visible in the same list.

What to look for:
- **Inconsistencies:** the same meaning written in different ways, or answers that contradict our filters.
- **Non-informative answers:** e.g. "Other", "I don't know", "Prefer not to say". They are not missing, but they tell us little.
- **Rare categories:** groups with very few respondents, which may need to be grouped later.
- **Formatting:** very long labels or special characters.

`Country` was already checked in Section 8, and `LanguageHaveWorkedWith` is a multi-select column that will be handled separately, so both are left out here.

In [28]:
categorical_cols = ["DevType", "EdLevel", "OrgSize", "RemoteWork", "Industry", "ICorPM", "Age"]

# Print every category and its count for each column; to_string() avoids cut-off output
for col in categorical_cols:
    print(f"--- {col} ---")
    print(df_clean[col].value_counts(dropna=False).to_string(), "\n")

--- DevType ---
DevType
Developer, full-stack                            2637
Developer, back-end                              1580
Architect, software or solutions                  591
Developer, desktop or enterprise applications     456
Developer, front-end                              374
Developer, embedded applications or devices       372
Developer, mobile                                 261
DevOps engineer or professional                   231
Engineering manager                               170
Data engineer                                     162
Other (please specify):                           125
AI/ML engineer                                    109
Data scientist                                     97
Academic researcher                                85
Developer, game or graphics                        72
Senior executive (C-suite, VP, etc.)               70
Cloud infrastructure engineer                      68
Developer, QA or test                              53
Appl

In [29]:
# The multi-select column: several answers in one cell, separated by ";"
df_clean["LanguageHaveWorkedWith"].head()

1                                                           Java
27                                            C#;Java;Python;SQL
35    Bash/Shell (all shells);HTML/CSS;JavaScript;SQL;TypeScript
38                                    C#;HTML/CSS;SQL;TypeScript
43                                                 C#;C++;Python
Name: LanguageHaveWorkedWith, dtype: object

**Observations:**

*Audit table (9.2)*
- **Data types:** numeric columns are stored correctly as numbers. `WorkExp` and `YearsCode` are `float64` although they hold whole numbers, because pandas stores `NaN` as a float; this does not affect the analysis. `Age`, `OrgSize` and `EdLevel` are stored as text but are **ordered** categories; their natural order will be defined for EDA and encoding.
- **Unique values:** `ConvertedCompYearly` has only 1,843 different values for 7,709 rows, because people report round salaries in their own currency, which convert to identical USD values. `LanguageHaveWorkedWith` has 4,397 different combinations, so it cannot be encoded as it is and must be split into individual languages.
- **Formatting:** multi-select answers are separated by `;`, several labels are very long (e.g. `RemoteWork`), and some labels contain a typographic apostrophe (`’` in "Master’s", "I don’t know"). Filtering with a straight apostrophe (`'`) would silently match no rows, so labels must be copied from the data.

*Duplicates (9.3)*
- No duplicate `ResponseId`. Three pairs have identical answers in all analysis columns; the second copy of each pair was removed (see the decision above). 7,706 rows remain.

*Categorical values (9.4), after removing the duplicates*
- **Answers that contradict the filters:** 5 respondents describe their job as "Student" (`DevType`) and 16 say they are a freelancer (`OrgSize`), although they said they are employed professional developers. These 21 rows (0.3%) do not match the target user and will be removed.
- **Non-informative answers:** "Other" in `DevType` (125), `EdLevel` (70) and `Industry` (602, about 8%), "I don’t know" in `OrgSize` (66) and "Prefer not to say" in `Age` (3). Large groups are kept as a separate category; very small ones will be treated as missing.
- **Rare categories:** about half of the 31 `DevType` categories have fewer than 30 respondents. Roles will be grouped into meaningful families during the cleaning of categorical columns.
- **Imbalance:** about 88% of respondents are individual contributors and 12% are people managers. This reflects reality and will be considered in the group comparison tests.

## 10. Cleaning step 1: remove answers that contradict the scope

The audit in Section 9.4 found 21 respondents whose answers contradict our filters:
- **5 respondents** describe their job (`DevType`) as "Student".
- **16 respondents** describe their organisation size (`OrgSize`) as "Just me - I am a freelancer, sole proprietor, etc.".

All of them said they are professional developers (`MainBranch`) and employed (`Employment`).

**These answers are not necessarily wrong.** In real life a person can be both a student and an employee (for example a working student, which is common in Germany: three of the five "Student" rows are from Germany and four have no work experience). A contractor can also see themselves as both employed and freelance. So the question is not *"Are these answers wrong?"* but *"Do these people match our target user?"*

**Decision: remove these 21 rows.**

| Option | Pros | Cons |
|---|---|---|
| **Remove** | The data stays consistent with the scope defined in Section 6 (employed professional developers, freelancers excluded). The model does not learn from part-time student salaries. Two tiny categories (5 and 16 rows) disappear and need no special handling. | 21 rows lost (0.3% of the data). The model cannot be used for working students. |
| **Keep** | No data loss; real-life mixed situations are kept. | Contradicts our own scope decision. The two tiny categories still need grouping or another treatment. The model partly learns from people outside the target group. |

The effect on the results is very small either way, so the decision is mainly about **consistency**: the same target user definition is applied at every step of the project. In short: *these rows are removed because they do not match the target user, not because they are errors.*

In [ ]:
# Exact labels, copied from the output of Section 9.4
contradicting_devtype = "Student"
contradicting_orgsize = "Just me - I am a freelancer, sole proprietor, etc."

# Boolean mask: True for rows that contradict the scope
contradicts_scope = (
    (df_clean["DevType"] == contradicting_devtype)
    | (df_clean["OrgSize"] == contradicting_orgsize)
)

# Validation check: we expect 5 + 16 = 21 rows before removing anything
print("Student (DevType):  ", (df_clean["DevType"] == contradicting_devtype).sum())
print("Freelancer (OrgSize):", (df_clean["OrgSize"] == contradicting_orgsize).sum())
print("Rows to remove:      ", contradicts_scope.sum())

`|` means "or": a row is marked if **either** condition is true. Each condition is in its own brackets, because pandas needs them to combine conditions correctly.

The validation check prints the counts before removing anything. If a label were misspelled, the count would be 0 and we would notice it here, instead of silently removing nothing.

To remove the rows, we keep everything that is **not** marked: `~` reverses the mask (True becomes False and the other way round).

In [ ]:
rows_before = len(df_clean)

# Keep all rows that do NOT contradict the scope
df_clean = df_clean[~contradicts_scope].copy()

print("Rows before:", rows_before)
print("Rows after: ", len(df_clean))
print("Removed:    ", rows_before - len(df_clean))

## Next steps
- Investigate salary outliers and define a justified rule
- Check the experience columns (`WorkExp`, `YearsCode`), including the inconsistency found in Section 9.3
- Clean the categorical columns: shorten labels, group rare categories, handle non-informative answers